# 제16장: 의료 AI 거버넌스

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/back2zion/ai-governance-handbook/blob/main/notebooks/ch16.ipynb)

> **『AI 거버넌스 실무 지침서』** (곽두일) 실습 코드
>
> 이 노트북의 모든 코드는 Google Colab에서 바로 실행할 수 있습니다.

In [ ]:
# 필요 패키지 설치 (최초 1회)
!pip install -q fairlearn shap mlflow diffprivlib alibi alibi-detect evidently pandas scikit-learn matplotlib seaborn numpy

**PCCP 기반 의료 AI 변경 관리 시스템**

In [ ]:
from dataclasses import dataclass, field
from typing import List, Optional, Dict
from enum import Enum
from datetime import datetime
import numpy as np

class ChangeType(Enum):
    RETRAIN = "Model Retrain"
    ARCHITECTURE = "Architecture Change"
    FEATURE_ADD = "Feature Addition"
    THRESHOLD = "Threshold Adjustment"

class ApprovalStatus(Enum):
    PRE_APPROVED = "Pre-approved (Within PCCP)"
    REQUIRES_SUBMISSION = "Requires New Submission"
    BLOCKED = "Blocked - Outside Scope"

@dataclass
class PCCPBoundary:
    """FDA PCCP Predetermined Change Control Plan"""
    max_performance_drift: float = 0.05   # AUC drop limit
    allowed_change_types: List[ChangeType] = field(
        default_factory=lambda: [ChangeType.RETRAIN, ChangeType.THRESHOLD]
    )
    min_validation_size: int = 500
    required_subgroup_parity: float = 0.10  # Max disparity

### 실행

In [ ]:
class MedicalAIChangeController:
    """PCCP-based Change Management for SaMD"""
    def __init__(self, pccp: PCCPBoundary, baseline_auc: float):
        self.pccp = pccp
        self.baseline_auc = baseline_auc
        self.change_log = []

    def evaluate_change(self, change_type, new_auc,
                        validation_size, subgroup_disparity):
        # Step 1: Is change type pre-approved?
        if change_type not in self.pccp.allowed_change_types:
            return ApprovalStatus.REQUIRES_SUBMISSION

        # Step 2: Performance within boundary?
        drift = self.baseline_auc - new_auc
        if drift > self.pccp.max_performance_drift:
            return ApprovalStatus.BLOCKED

        # Step 3: Sufficient validation data?
        if validation_size < self.pccp.min_validation_size:
            return ApprovalStatus.REQUIRES_SUBMISSION

        # Step 4: Subgroup parity maintained?
        if subgroup_disparity > self.pccp.required_subgroup_parity:
            return ApprovalStatus.BLOCKED

        self.change_log.append({
            "timestamp": datetime.now().isoformat(),
            "change_type": change_type.value,
            "new_auc": new_auc, "status": "DEPLOYED"
        })
        return ApprovalStatus.PRE_APPROVED

**의료 AI 하위 그룹 성능 분석기**

In [ ]:
from dataclasses import dataclass
from typing import List, Dict, Tuple, Optional
import numpy as np
from collections import defaultdict

@dataclass
class PatientRecord:
    patient_id: str
    prediction: float      # Model predicted probability
    ground_truth: int      # 0 or 1
    age_group: str         # e.g., "0-18", "19-40", "41-65", "65+"
    sex: str               # "M", "F"
    ethnicity: str         # e.g., "Asian", "Black", "White", "Hispanic"
    device_type: str       # Imaging device model

### 실행

In [ ]:
class SubgroupPerformanceAnalyzer:
    """
    Comprehensive subgroup performance analysis
    for medical AI fairness assessment
    (FDA diversity requirements compliant)
    """
    def __init__(self, significance_threshold: float = 0.10,
                 min_subgroup_size: int = 30):
        self.significance_threshold = significance_threshold
        self.min_subgroup_size = min_subgroup_size
        self.records: List[PatientRecord] = []

    def add_records(self, records: List[PatientRecord]):
        self.records.extend(records)

    def compute_auc(self, predictions: List[float],
                    labels: List[int]) -> float:
        """Compute AUC-ROC using trapezoidal rule"""
        if len(set(labels)) < 2:
            return float('nan')
        paired = sorted(zip(predictions, labels), reverse=True)
        tp, fp, prev_tp, prev_fp = 0, 0, 0, 0
        auc = 0.0
        n_pos = sum(labels)
        n_neg = len(labels) - n_pos
        for _, label in paired:
            if label == 1:
                tp += 1
            else:
                fp += 1
                auc += (tp + prev_tp) / 2
            prev_tp, prev_fp = tp, fp
        return auc / (n_pos * n_neg) if n_pos * n_neg > 0 else 0.0

    def compute_metrics_at_threshold(
        self, predictions: List[float],
        labels: List[int], threshold: float = 0.5
    ) -> Dict[str, float]:
        """Compute sensitivity, specificity, PPV, NPV"""
        tp = sum(1 for p, l in zip(predictions, labels)
                 if p >= threshold and l == 1)
        fp = sum(1 for p, l in zip(predictions, labels)
                 if p >= threshold and l == 0)
        tn = sum(1 for p, l in zip(predictions, labels)
                 if p < threshold and l == 0)
        fn = sum(1 for p, l in zip(predictions, labels)
                 if p < threshold and l == 1)
        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
        specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
        ppv = tp / (tp + fp) if (tp + fp) > 0 else 0
        npv = tn / (tn + fn) if (tn + fn) > 0 else 0
        return {
            "sensitivity": sensitivity, "specificity": specificity,
            "ppv": ppv, "npv": npv,
            "tp": tp, "fp": fp, "tn": tn, "fn": fn,
        }

    def analyze_by_attribute(self, attribute: str) -> Dict:
        """Analyze performance across subgroups"""
        groups = defaultdict(list)
        for r in self.records:
            key = getattr(r, attribute, "Unknown")
            groups[key].append(r)

        results = {}
        for group_name, group_records in groups.items():
            if len(group_records) < self.min_subgroup_size:
                results[group_name] = {
                    "status": "INSUFFICIENT_DATA",
                    "n": len(group_records),
                    "min_required": self.min_subgroup_size,
                }
                continue
            preds = [r.prediction for r in group_records]
            labels = [r.ground_truth for r in group_records]
            auc = self.compute_auc(preds, labels)
            metrics = self.compute_metrics_at_threshold(preds, labels)
            results[group_name] = {
                "n": len(group_records),
                "prevalence": sum(labels) / len(labels),
                "auc": round(auc, 4),
                **{k: round(v, 4) for k, v in metrics.items()
                   if isinstance(v, float)},
            }
        return results

    def compute_disparity_report(self) -> Dict:
        """Generate comprehensive disparity report"""
        report = {"attributes": {}, "alerts": []}
        for attr in ["age_group", "sex", "ethnicity", "device_type"]:
            analysis = self.analyze_by_attribute(attr)
            aucs = {
                k: v["auc"] for k, v in analysis.items()
                if isinstance(v.get("auc"), float)
                and not np.isnan(v["auc"])
            }
            if len(aucs) >= 2:
                max_auc = max(aucs.values())
                min_auc = min(aucs.values())
                disparity = max_auc - min_auc
                worst_group = min(aucs, key=aucs.get)
                best_group = max(aucs, key=aucs.get)

                report["attributes"][attr] = {
                    "subgroups": analysis,
                    "max_disparity": round(disparity, 4),
                    "best_group": best_group,
                    "worst_group": worst_group,
                }
                if disparity > self.significance_threshold:
                    report["alerts"].append({
                        "attribute": attr,
                        "disparity": round(disparity, 4),
                        "message": (
                            f"AUC disparity of {disparity:.4f} "
                            f"between {best_group} and "
                            f"{worst_group} exceeds threshold "
                            f"{self.significance_threshold}"
                        ),
                        "action": "REQUIRES_MITIGATION",
                    })

        report["overall_status"] = (
            "FAIL" if report["alerts"] else "PASS"
        )
        return report

    def generate_fda_summary(self) -> Dict:
        """Generate FDA-compliant diversity summary"""
        demographics = {
            "total_patients": len(self.records),
            "age_distribution": {},
            "sex_distribution": {},
            "ethnicity_distribution": {},
        }
        for attr, key in [
            ("age_group", "age_distribution"),
            ("sex", "sex_distribution"),
            ("ethnicity", "ethnicity_distribution"),
        ]:
            counts = defaultdict(int)
            for r in self.records:
                counts[getattr(r, attr, "Unknown")] += 1
            demographics[key] = {
                k: {"count": v, "percentage": f"{v/len(self.records):.1%}"}
                for k, v in counts.items()
            }
        disparity = self.compute_disparity_report()
        return {
            "demographics": demographics,
            "performance_analysis": disparity,
            "compliance": disparity["overall_status"],
        }

**DICOM 의료영상 비식별화 파이프라인**

In [ ]:
from dataclasses import dataclass
from typing import List, Dict, Set
from datetime import datetime

@dataclass
class DICOMDeidentificationRule:
    """HIPAA Safe Harbor + DICOM PS3.15 Annex E"""
    # 18 HIPAA identifiers mapped to DICOM tags
    REMOVE_TAGS: List[str] = None
    HASH_TAGS: List[str] = None
    DATE_SHIFT_DAYS: int = 0

    def __post_init__(self):
        self.REMOVE_TAGS = [
            "(0010,0010)",  # Patient Name
            "(0010,0020)",  # Patient ID
            "(0010,0030)",  # Birth Date
            "(0010,1001)",  # Other Patient Names
            "(0008,0050)",  # Accession Number
            "(0008,0080)",  # Institution Name
            "(0008,0081)",  # Institution Address
            "(0008,1070)",  # Operators Name
            "(0010,1040)",  # Patient Address
            "(0010,2154)",  # Patient Telephone Numbers
            "(0010,0040)",  # Patient Sex (context-dependent)
            "(0008,0090)",  # Referring Physician Name
            "(0008,1050)",  # Performing Physician Name
        ]
        self.HASH_TAGS = [
            "(0020,000D)",  # Study Instance UID
            "(0020,000E)",  # Series Instance UID
            "(0008,0018)",  # SOP Instance UID
        ]

### 실행

In [ ]:
class MedicalImageDeidentifier:
    """DICOM De-identification Pipeline"""
    def __init__(self, rules: DICOMDeidentificationRule):
        self.rules = rules
        self.audit_log = []

    def deidentify(self, dataset, study_id):
        """Process single DICOM file"""
        result = {"study_id": study_id, "actions": []}

        # Step 1: Remove direct identifiers
        for tag in self.rules.REMOVE_TAGS:
            if tag in str(dataset):
                result["actions"].append(f"REMOVED: {tag}")

        # Step 2: Hash UIDs for linkability
        for tag in self.rules.HASH_TAGS:
            result["actions"].append(f"HASHED: {tag}")

        # Step 3: Date shifting
        if self.rules.DATE_SHIFT_DAYS != 0:
            result["actions"].append(
                f"DATE_SHIFTED: {self.rules.DATE_SHIFT_DAYS} days"
            )

        # Step 4: Pixel data scrubbing (burned-in text)
        result["actions"].append("PIXEL_SCRUB: OCR-based PHI detection")

        # Step 5: Private tag removal
        result["actions"].append("PRIVATE_TAGS: All removed")

        # Audit trail
        self.audit_log.append({
            "timestamp": datetime.now().isoformat(),
            "study_id": study_id,
            "tags_removed": len(self.rules.REMOVE_TAGS),
            "tags_hashed": len(self.rules.HASH_TAGS),
        })
        return result

    def validate_deidentification(self, dataset) -> Dict:
        """Verify completeness of de-identification"""
        remaining_phi = []
        # Check for remaining PHI patterns
        phi_patterns = [
            "Patient", "Name", "Birth", "Address",
            "Phone", "SSN", "Insurance"
        ]
        for pattern in phi_patterns:
            if pattern.lower() in str(dataset).lower():
                remaining_phi.append(pattern)
        return {
            "is_clean": len(remaining_phi) == 0,
            "remaining_phi_indicators": remaining_phi,
            "recommendation": "SAFE" if not remaining_phi
                             else "REQUIRES_REVIEW",
        }

    def generate_audit_report(self):
        return {
            "total_processed": len(self.audit_log),
            "compliance": "HIPAA Safe Harbor Method",
            "details": self.audit_log
        }